# JoyCaption API Test

Test the JoyCaption GGUF node for image captioning.

## Prerequisites

1. ComfyUI container rebuilt with JoyCaption node
2. GGUF model downloaded to `/home/warby/ComfyUI/models/LLM/GGUF/llama-joycaption-beta-one-hf-llava.Q4_K_M.gguf`
3. ComfyUI running on localhost:8188

## Model Details

- Model: JoyCaption Beta One (Llama 3.1 8B based)
- Quantization: Q4_K_M (5.0 GB)
- VRAM: 5-6GB required
- Node: JC_GGUF from 1038lab/ComfyUI-JoyCaption

In [ ]:
import json
import time
import requests
import uuid
import websocket
from pathlib import Path
import os

In [ ]:
# Configuration
COMFY_URL = "http://localhost:8188"
client_id = str(uuid.uuid4())

# Test image path (update with your test image)
TEST_IMAGE = "test_image.jpg"  # Update this path

## Helper Functions

In [ ]:
def upload_image(image_path):
    """Upload an image to ComfyUI"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    with open(image_path, 'rb') as f:
        files = {'image': (os.path.basename(image_path), f, 'image/png')}
        response = requests.post(f"{COMFY_URL}/upload/image", files=files)
        response.raise_for_status()
        return response.json()['name']

def queue_prompt(prompt):
    """Queue a workflow prompt"""
    prompt_id = str(uuid.uuid4())
    payload = {"prompt": prompt, "client_id": client_id, "prompt_id": prompt_id}
    response = requests.post(f"{COMFY_URL}/prompt", json=payload)
    response.raise_for_status()
    return prompt_id

def get_history(prompt_id):
    """Get execution history for a prompt"""
    response = requests.get(f"{COMFY_URL}/history/{prompt_id}")
    response.raise_for_status()
    return response.json()

def wait_for_completion(ws, prompt_id):
    """Wait for workflow execution to complete via WebSocket"""
    while True:
        out = ws.recv()
        if isinstance(out, str):
            message = json.loads(out)
            if message['type'] == 'executing':
                data = message['data']
                if data['node'] is None and data['prompt_id'] == prompt_id:
                    return True
            elif message['type'] == 'execution_error':
                print(f"Execution error: {message}")
                return False

## JoyCaption Workflow

In [ ]:
# JoyCaption workflow template
workflow_template = {
    "1": {
        "class_type": "LoadImage",
        "inputs": {
            "image": "placeholder.png"
        }
    },
    "2": {
        "class_type": "JC_GGUF",
        "inputs": {
            "image": ["1", 0],
            "model": "llama-joycaption-beta-one-hf-llava.Q4_K_M.gguf",
            "processing_mode": "Auto",
            "prompt_style": "Descriptive",
            "caption_length": "any",
            "memory_management": "Keep in Memory"
        }
    }
}

print("Workflow template loaded")
print(json.dumps(workflow_template, indent=2))

## Run Caption Test

In [ ]:
# Upload test image
print(f"Uploading image: {TEST_IMAGE}")
uploaded_filename = upload_image(TEST_IMAGE)
print(f"Uploaded as: {uploaded_filename}")

# Update workflow with uploaded image
workflow = workflow_template.copy()
workflow["1"]["inputs"]["image"] = uploaded_filename

In [ ]:
# Connect WebSocket and execute
print("Connecting to ComfyUI...")
ws = websocket.WebSocket()
ws.connect(f"ws://localhost:8188/ws?clientId={client_id}")

print("Queuing workflow...")
prompt_id = queue_prompt(workflow)
print(f"Prompt ID: {prompt_id}")

print("Waiting for completion...")
success = wait_for_completion(ws, prompt_id)
ws.close()

if success:
    print("Execution completed successfully")
else:
    print("Execution failed")

In [ ]:
# Retrieve caption from history
history = get_history(prompt_id)
if prompt_id in history:
    outputs = history[prompt_id].get('outputs', {})
    
    # JC_GGUF node is node "2"
    if '2' in outputs:
        node_output = outputs['2']
        if 'text' in node_output:
            caption = node_output['text'][0]
            print("\n" + "="*80)
            print("Generated Caption:")
            print("="*80)
            print(caption)
            print("="*80)
        else:
            print("No text output found")
            print(f"Available outputs: {node_output.keys()}")
    else:
        print("Node 2 not found in outputs")
        print(f"Available nodes: {outputs.keys()}")
else:
    print(f"Prompt ID {prompt_id} not found in history")

## Test Different Caption Styles

Available prompt styles:
- Descriptive
- Casual
- Straightforward
- Tags
- Technical
- Artistic

In [ ]:
# Test different caption styles
styles = ["Descriptive", "Casual", "Tags", "Technical"]

results = {}

ws = websocket.WebSocket()
ws.connect(f"ws://localhost:8188/ws?clientId={client_id}")

for style in styles:
    print(f"\nTesting style: {style}")
    
    workflow = workflow_template.copy()
    workflow["1"]["inputs"]["image"] = uploaded_filename
    workflow["2"]["inputs"]["prompt_style"] = style
    
    prompt_id = queue_prompt(workflow)
    wait_for_completion(ws, prompt_id)
    
    history = get_history(prompt_id)
    if prompt_id in history and '2' in history[prompt_id].get('outputs', {}):
        caption = history[prompt_id]['outputs']['2'].get('text', [''])[0]
        results[style] = caption
        print(f"Caption: {caption[:100]}...")

ws.close()

# Display all results
print("\n" + "="*80)
print("Caption Style Comparison")
print("="*80)
for style, caption in results.items():
    print(f"\n{style}:")
    print("-" * 40)
    print(caption)